### RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_3540/582141364.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
/workspaces/RAG-Application/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the pdf's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF Files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    #Find all the PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()[:40]

            #Add Source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} Pages")
        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents processed: {len(all_documents)}")
    return all_documents

#Process all the pdf's in the directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files in ../data

Processing: book_on_building_agents.pdf
 Loaded 40 Pages

Processing: paper.pdf
 Loaded 14 Pages

Total documents processed: 54


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Haggai Roitman', 'doi': 'https://doi.org/10.48550/arXiv.2606.24937', 'license': 'http://creativecommons.org/licenses/by-sa/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': "The Hitchhiker's Guide to Agentic AI: From Foundations to Systems", 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2606.24937v1', 'source': '../data/pdf/book_on_building_agents.pdf', 'total_pages': 603, 'page': 0, 'page_label': '1', 'source_file': 'book_on_building_agents.pdf', 'file_type': 'pdf'}, page_content='The Hitchhiker’s Guide to Agentic AI\nFrom Foundations to Systems\nHaggai Roitman\n2026\nVersion 1.2.2\narXiv:2606.24937v1  [cs.AI]  22 Jun 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Haggai Roitman', 'doi': 'http

In [4]:
### Text Splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} into {len(split_docs)} chunks")
    
    
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
print(chunks)

Split 54 into 231 chunks

Example chunk:
Content: The Hitchhiker’s Guide to Agentic AI
From Foundations to Systems
Haggai Roitman
2026
Version 1.2.2
arXiv:2606.24937v1  [cs.AI]  22 Jun 2026...
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Haggai Roitman', 'doi': 'https://doi.org/10.48550/arXiv.2606.24937', 'license': 'http://creativecommons.org/licenses/by-sa/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': "The Hitchhiker's Guide to Agentic AI: From Foundations to Systems", 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2606.24937v1', 'source': '../data/pdf/book_on_building_agents.pdf', 'total_pages': 603, 'page': 0, 'page_label': '1', 'source_file': 'book_on_building_agents.pdf', 'file_type': 'pdf'}
[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Haggai R

### Embedding and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace Model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded model succesfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
        
        Returns:
            Numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:02<00:00, 42.74it/s]


Loaded model succesfully. Embedding dimension: 384


### Vector Store

In [8]:
class VectorStore:
    """Manages document embedding in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the Vector Store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the ChromaDB data
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize the ChromaDB client and collection"""
        try:
            #Create persistent chromaDB client
            print(f"Initializing ChromaDB client with persist directory: {self.persist_directory}")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            #Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF Document Embeddings for RAG"}
            )
        except Exception as e:
            print(f"Error initializing ChromaDB vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of Langchain documents
            embeddings: corresponding embedding for the documents
        """

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to the vector store...")

        #Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate Unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            #Prepare Metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            #Document Content
            documents_text.append(doc.page_content)

            #Embedding
            embeddings_list.append(embedding.tolist())

        #Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection '{self.collection_name}': {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore


Initializing ChromaDB client with persist directory: ../data/vector_store


In [9]:
chunks

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Haggai Roitman', 'doi': 'https://doi.org/10.48550/arXiv.2606.24937', 'license': 'http://creativecommons.org/licenses/by-sa/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': "The Hitchhiker's Guide to Agentic AI: From Foundations to Systems", 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2606.24937v1', 'source': '../data/pdf/book_on_building_agents.pdf', 'total_pages': 603, 'page': 0, 'page_label': '1', 'source_file': 'book_on_building_agents.pdf', 'file_type': 'pdf'}, page_content='The Hitchhiker’s Guide to Agentic AI\nFrom Foundations to Systems\nHaggai Roitman\n2026\nVersion 1.2.2\narXiv:2606.24937v1  [cs.AI]  22 Jun 2026'),
 Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Haggai Roitman', 'doi': 'http

In [10]:
### Convert the text to embeddings

texts=[doc.page_content for doc in chunks]

##Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

##Store into the vector database
vectorstore.add_documents(chunks, embeddings)


Generating embeddings for 231 texts...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Batches: 100%|██████████| 8/8 [00:28<00:00,  3.59s/it]


Generated embeddings with shape: (231, 384)
Adding 231 documents to the vector store...
Successfully added 231 documents to the vector store.
Total documents in collection 'pdf_documents': 924


## Retriever Pipeline from VectorStore

In [11]:
class RAGRetriever:
    """Handles query-base retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG Retriever

        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager: manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query:str, top_k:int=5, score_threshold:float=0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a given query

        Args:
            query: The input query string
            top_k: Number of top documents to retrieve
            score_threshold: Minimum similarity score for retrieved documents

        Returns:
            List of retrieved documents with their metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
        # Retrieve top-k similar documents
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()], 
                n_results=top_k
            )

            #Process Results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, doc_content, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    #Convert distance to similarity score(ChromaDB uses cosine distance, so similarity = 1 - distance)
                    similarity_score = 1 - distance  # Convert distance to similarity score

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": doc_content,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank":i+1
                        })
                    
                print(f"Retrieved {len(retrieved_docs)} documents after applying score threshold filter.")
            
            else:
                print("No documents retrieved from the vector store.")
                return []
            
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [12]:
rag_retriever

In [13]:
rag_retriever.retrieve(" Question Answering (QA) with Language Models (LLMs)")

Retrieving documents for query: ' Question Answering (QA) with Language Models (LLMs)'
Top K: 5, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.21it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents after applying score threshold filter.


[{'id': 'doc_7054ecfe_179',
  'content': 'Augmented Generation (CRAG) introduces evaluators to assess document quality and refine retrieval\nactions Yan et al. [2024]. Unlike RAG, RAG-end2end Siriwardhana et al. [2023] jointly trains\nretrievers and generators, enhancing open-domain question answering by updating all components,\nincluding external knowledge bases.\n2.2 Question Answering (QA) with Language Models (LLMs)\nXu et al. [2023] democratizes advanced chat models, enhancing Llama’s dialogue performance\nthrough fine-tuning and Self-Distillation with Feedback (SDF) further improves its capabilities.\nComparing RAG and fine-tuning with synthetic data, fine-tuning shows significant performance\nimprovements Soudani et al. [2024]. ChatQA Liu et al. [2024] surpasses GPT-4 in retrieval-augmented\ngeneration and conversational QA. The Chain-of-Action (CoA) framework Pan et al. [2024] addresses\ncomplex questions by decomposing them into reasoning chains, effectively tackling hallucin

### Integrate Vectordb Context Pipeline with LLM output

In [14]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
load_dotenv()

# Initialize the Gemini Model within LangChain
# By default, it looks for the 'GOOGLE_API_KEY' environment variable.
# You can also use 'gemini-2.5-pro' depending on your performance needs.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
)

## 2. Simple (RAG) function : retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ### retrieve the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n].\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    
    ## generate the answer using Gemini via LangChain
    prompt = f"""Use the following context to answer the question concisely.
    Context: {context}
    Question: {query}
    Answer:"""

    # Pass the formatted prompt to the Gemini model
    response = llm.invoke([prompt])
    return response.content


In [15]:
answer = rag_simple("Question Answering (QA) with Language Models (LLMs)", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'Question Answering (QA) with Language Models (LLMs)'
Top K: 3, Score Threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 67.69it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold filter.


Xu et al. [2023] enhanced Llama's dialogue performance through fine-tuning and Self-Distillation with Feedback (SDF). Soudani et al. [2024] found that fine-tuning with synthetic data significantly improves performance over RAG. ChatQA Liu et al. [2024] surpasses GPT-4 in retrieval-augmented generation and conversational QA. The Chain-of-Action (CoA) framework Pan et al. [2024] addresses complex questions by decomposing them into reasoning chains to tackle hallucinations.


### Enhance RAG Pipeline featues

In [16]:
# ---- Enhanced RAG Pipeline features---------------

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features
    - Returns answer, sources, confidence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer':'No relevant context found', 'sources':[], 'confidence':0.0, 'context': ''}
    
    #prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source','unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    #Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext:\n{context}\n\nQuestion: {query}\n\nAnswer: """
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer' : response.content,
        'sources' : sources,
        'confidence' : confidence
    }
    if return_context:
        output['context']=context
    else:
        output['context']='Context display is not selected'
    return output

#Example usage:
result = rag_advanced("(QA) with Language Models (LLMs)", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer: ", result['answer'])
print("sources: ", result['sources'])
print("Confidence: ", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: '(QA) with Language Models (LLMs)'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 68.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold filter.


Answer:  The provided context describes the foundational architecture and optimization methods for Large Language Models (LLMs), including their pipeline from text to tokens to representations and back to text. It does not contain information about Question Answering (QA) specifically with LLMs.
sources:  [{'source': 'book_on_building_agents.pdf', 'page': 34, 'score': 0.14222902059555054, 'preview': 'Chapter 1\nLLM Architecture and Optimization\nMethods\nThis section covers the foundational architecture of large language models and the key optimization\ntechniques that make training and inference efficient. Topics are ordered as a curriculum: we begin\nwith the transformer itself, then cover how to t...'}, {'source': 'book_on_building_agents.pdf', 'page': 34, 'score': 0.14222902059555054, 'preview': 'Chapter 1\nLLM Architecture and Optimization\nMethods\nThis section covers the foundational architecture of large language models and the key optimization\ntechniques that make training an

In [18]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("(QA) with Language Models (LLMs)", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: '(QA) with Language Models (LLMs)'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.55it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents after applying score threshold filter.
Streaming answer:
Use the following context to answer the question concisely.
Context:
Chapter 1
LLM Architecture and Optimization
Methods
This section covers the foundational ar

chitecture of large language models and the key optimization
techniques that make training and inference efficient. Topics are ordered as a curriculum: we begin
with the transformer itself, then cover how to train it efficiently, how to adapt it cheaply, how to
compress it, how to scale it, and how to accelerate its inference.
1.1 How LLMs Work: An Intuitive Overview
Before diving into architectural details, let us build intuition for how a large language model transforms
text into text. The entire process follows a simple pipeline:text→ tokens→ representations→
tokens→text.
Raw
Text Tokenizer Token
IDs
Embedding
Layer
Transformer
Layers (×L)
Vocab
Logits Decode Output
Text
autoregressive loop (append token to input)
Figure 1.1:The LLM pipeline: text is tokenized into subword units, converted to integer IDs, embedded as

Chapter 1
LLM Architecture and Optimization
Methods
This section covers the foundational architecture of large language models and the key optimization
techniques that